# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset's Croissant schema is available at:
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
First, load the dataset metadata and explore key summary information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset Title:", metadata.name)
print("Description:", metadata.description)
print("Authors (@id):", [a['@id'] if isinstance(a, dict) and '@id' in a else a for a in getattr(metadata, 'author', [])])
print("Record Set IDs:", metadata.recordSet if hasattr(metadata, 'recordSet') else [])
print("Distribution (@id):", [d['@id'] if isinstance(d, dict) and '@id' in d else d for d in getattr(metadata, 'distribution', [])])

## 2. Data Overview
Review record set, field, and column `@id`s present in the dataset. All references use the `@id` in keeping with the Croissant schema.

In [ ]:
# Retrieve all available record sets and their @id
record_sets = dataset.record_sets
print("Record Sets and their @id:")
for rs in record_sets:
    print("- Name:", rs.name, "@id:", rs['@id'])
    # List fields for each record set
    print("  Fields:")
    for field in getattr(rs, 'fields', []):
        field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
        print("    -", field.get('name', 'N/A'), "(@id:", field_id, ")")
    # List columns for each record set, if present
    if hasattr(rs, 'columns'):
        print("  Columns:")
        for col in rs.columns:
            col_id = col['@id'] if isinstance(col, dict) and '@id' in col else col
            print("    -", col.get('name', 'N/A'), "(@id:", col_id, ")")


## 3. Data Extraction
Extract data from each record set and load it into a DataFrame. We reference the record set `@id` and field `@id` as obtained above.

In [ ]:
# Get all record set @id values
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"Loaded record set {rsid}: shape = {df.shape}")
# Preview columns from the first record set
if record_set_ids:
    first_rs = record_set_ids[0]
    print("Columns in", first_rs, ":", dataframes[first_rs].columns.tolist())
    dataframes[first_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic processing: filtering, normalizing numeric values, and grouping data. All field references use their `@id`.

For demonstration, we pick a numeric field and a group field using one of the record set column `@id`s from the overview above. Replace `<numeric_field_id>` and `<group_field_id>` with actual field `@id`s available in your dataset.

In [ ]:
# Choose first record set and candidate fields
record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[record_set_id] if record_set_id else pd.DataFrame()
# List available columns
print("Available columns:", df.columns.tolist())

# Example field IDs (replace with actual IDs from your schema or preview above)
# For demonstration, we will guess typical column names
# Let's assume 'age_at_diagnosis' and 'infertility' are column @id in first record set
numeric_field_id = 'age_at_diagnosis'  # Replace with actual @id from your dataset
group_field_id = 'infertility'        # Replace with actual @id from your dataset

# Only process if fields exist
if numeric_field_id in df.columns and group_field_id in df.columns:
    threshold = 20  # Example threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by categorical/group field
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize numeric distributions and relationships. For example, display age distribution or compare hormone levels by phenotype, referencing the relevant fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of numeric field if available
if numeric_field_id in df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} (@id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Scatter plot numeric field vs. another
other_numeric_id = 'height'  # Example, use correct @id
if numeric_field_id in df.columns and other_numeric_id in df.columns:
    plt.figure(figsize=(6,4))
    sns.scatterplot(x=df[numeric_field_id], y=df[other_numeric_id])
    plt.title(f"Scatter plot: {numeric_field_id} vs {other_numeric_id} (@id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel(other_numeric_id)
    plt.show()

## 6. Conclusion

We have loaded the FAIR^2 dataset using `mlcroissant`, explored its structure and fields via the Croissant schema `@id`, extracted and processed tabular data, and visualized key characteristics such as age at diagnosis and phenotype relationships. This approach ensures reproducible and standards-compliant referencing to all dataset entities for further clinical or research analysis.

Remember: Each record set, field, and column was referenced using its Croissant `@id` per FAIR practice.